In [20]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/adult-census-income")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'adult-census-income' dataset.
Path to dataset files: /kaggle/input/adult-census-income


In [21]:
import pandas as pd
import numpy as np


In [22]:
import os
import glob

csv_files = glob.glob(os.path.join(path, '*.csv'))

if csv_files:
    # Assuming the first CSV file found is the main one
    file_to_read = csv_files[0]
    df = pd.read_csv(file_to_read)
else:
    # Handle the case where no CSV files are found
    raise FileNotFoundError(f"No CSV files found in the directory: {path}")

In [23]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


<h3 style = "font-family: Comic Sans MS;background-color:#7DF0A5	"> Observation: </h3>

* The dataset contains absolutely **no null values**!
* Age, Final Weight, Education Number, Capital Gain, Capital Loss and Hours Per Week are integer columns.
* There are no Float Datatypes in the dataset.
* Workclass, Education, Marital Status, Occupation, Relationship, Race, Sec, Native Country and Income are of object datatypes.
* Although the dataset does not contain any null values, a closer look (see cell 3) tells us that there are a lot of **'?'** values in our dataset. We will have to **replace** those values!

In [25]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education.num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital.gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital.loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours.per.week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


<h3 style = "font-family: Comic Sans MS;background-color:#7DF0A5	"> Observation: </h3>

* The minimum and maximum age of people in the dataset is 17 and 90 years respectively, while the average age is 37.
* The minimum and maximum years spent on education is 1 and 16 respectively, whereas the mean education level is 10 years.
* While the minimum and average capital gain is 0, maximum is 99999. This seems odd, maybe some error within the data collection.
* The number of hours spent per week varies between 1 to 99 and the average being 40 hours.

## Problem 1 (Gaps): missing values are written as "?"

The dataset looks complete. It is not.

In [28]:
# isna() sees nothing, because the missing values were written as the string "?"
print("Nulls reported by isna():", df.isna().sum().sum())

q = (df == "?").sum()
print()
print("Columns that hold the ? marker:")
print(q[q > 0])

rows_hit = int((df == "?").any(axis=1).sum())
print()
print("Cells affected:", int(q.sum()))
print("Rows affected :", rows_hit, "=", round(100 * rows_hit / len(df), 1), "percent of the dataset")

# Do workclass and occupation go missing on the same rows?
both     = int(((df["workclass"] == "?") & (df["occupation"] == "?")).sum())
wc_only  = int(((df["workclass"] == "?") & (df["occupation"] != "?")).sum())
occ_only = int(((df["occupation"] == "?") & (df["workclass"] != "?")).sum())
print()
print("Both workclass and occupation missing:", both)
print("Only workclass missing               :", wc_only)
print("Only occupation missing              :", occ_only)

print()
print("workclass values on the rows where only occupation is missing:")
print(df.loc[(df["occupation"] == "?") & (df["workclass"] != "?"), "workclass"].value_counts())

Nulls reported by isna(): 0

Columns that hold the ? marker:
workclass         1836
occupation        1843
native.country     583
dtype: int64

Cells affected: 4262
Rows affected : 2399 = 7.4 percent of the dataset

Both workclass and occupation missing: 1836
Only workclass missing               : 0
Only occupation missing              : 7

workclass values on the rows where only occupation is missing:
workclass
Never-worked    7
Name: count, dtype: int64


**Observation**

- `isna()` reports **0 nulls**, but 4,262 cells hold `?` across 2,399 rows, which is 7.4% of the dataset. Because the missingness was written as a string, every standard null check passes this file as clean.
- Three columns are affected: `occupation` (1,843), `workclass` (1,836) and `native.country` (583).
- `workclass` and `occupation` are missing on **exactly the same 1,836 rows**, and there is no row where only `workclass` is blank. That is one non-response event copied into two fields, not two independent problems.
- The 7 rows where only `occupation` is missing are all `Never-worked`. Those people genuinely have no occupation, so a blanket replacement of `?` with null would destroy a real value.

**Why it matters:** any validation, imputation or completeness check built on `isna()` passes this dataset. The `?` then survives into encoding as a spurious category and corrupts every group-by on those three columns.

## Problem 2 (Strange values): capital.gain uses 99999 as a placeholder

The maximum of 99999 spotted in `describe()` above is worth a closer look.

In [31]:
cg = df["capital.gain"]

print("Five largest distinct values:", [int(v) for v in sorted(cg.unique())[-5:]])
print()
print("Rows sitting at 99999 :", int((cg == 99999).sum()))
print("Next value below it   :", int(cg[cg < 99999].max()))
print("Empty gap in between  :", int(99999 - cg[cg < 99999].max()))

print()
print("Income bracket of those 99999 rows:")
print(df.loc[cg == 99999, "income"].value_counts())

print()
print("Mean with 99999    :", round(cg.mean(), 2))
print("Mean without 99999 :", round(cg[cg != 99999].mean(), 2))
print("Median             :", cg.median())

# The zeros are real data, not breakage
print()
print("capital.gain zeros :", int((cg == 0).sum()), "=", round(100 * (cg == 0).mean(), 1), "percent")
print("capital.loss max   :", int(df["capital.loss"].max()), "- smooth distribution, no gap, so not a sentinel")

Five largest distinct values: [25236, 27828, 34095, 41310, 99999]

Rows sitting at 99999 : 159
Next value below it   : 41310
Empty gap in between  : 58689

Income bracket of those 99999 rows:
income
>50K    159
Name: count, dtype: int64

Mean with 99999    : 1077.65
Mean without 99999 : 592.23
Median             : 0.0

capital.gain zeros : 29849 = 91.7 percent
capital.loss max   : 4356 - smooth distribution, no gap, so not a sentinel


**Observation**

- 99999 appears on **159 rows**. The next value down is **41,310**, leaving a gap of 58,689 that contains no observations at all. Real income distributions decay continuously, so this is a top-code placeholder and not a sum of money.
- **All 159 rows are `>50K`**, with no exceptions. That is exactly what censoring at a high threshold produces.
- The placeholder drags the mean up to **1,077.65** against a median of **0**. Removing it drops the mean to about 592. Every mean, standard deviation and correlation on this column is distorted.
- The all-zero quartiles are **not** a defect. 91.7% of `capital.gain` is genuinely zero because most respondents have no capital gains. `capital.loss` peaks at 4,356 inside a smooth distribution, so it is not a sentinel.

**Why it matters:** aggregate `capital.gain` by occupation and the placeholder inflates whichever groups contain it, so any ranking built on that average is really a ranking of the encoding.

## Problem 3 (Duplicates): 24 identical rows, and no primary key to judge them

The duplicate count above is only half the problem.

In [32]:
print("Exact duplicate rows       :", int(df.duplicated().sum()))
print("Distinct duplicate patterns:", len(df[df.duplicated(keep=False)].drop_duplicates()))

# Is there any column that could serve as a primary key?
print()
print("Columns that look like an identifier:", [c for c in df.columns if "id" in c.lower()])

counts = df["fnlwgt"].value_counts()
print()
print("fnlwgt unique values   :", df["fnlwgt"].nunique(), "of", len(df), "rows")
print("Rows sharing an fnlwgt :", int(counts[counts > 1].sum()))
top = counts[counts == counts.max()]
print("Most repeats by one value:", int(counts.max()), "times")
print("Values tied at that count:", len(top), "->", [int(x) for x in sorted(top.index)])

Exact duplicate rows       : 24
Distinct duplicate patterns: 23

Columns that look like an identifier: []

fnlwgt unique values   : 21648 of 32561 rows
Rows sharing an fnlwgt : 17231
Most repeats by one value: 13 times
Values tied at that count: 3 -> [123011, 164190, 203488]


**Observation**

- 24 rows are exact duplicates, forming 23 distinct patterns.
- There is **no identifier column of any kind**. No `id`, no `person_id`, nothing.
- `fnlwgt` cannot stand in for one: only 21,648 of 32,561 values are unique, **17,231 rows share a value** with at least one other row, and three different values tie for the most repeated, at 13 rows each. This is not a flaw in `fnlwgt`. It is a Current Population Survey weight, so similar respondents receive similar weights by design and collision is its intended behaviour.
- Without a key we cannot tell a data-entry duplicate from two different people who happen to match on all 14 attributes. Both are plausible, because the duplicated rows sit in low-variance profiles where genuine collisions are most likely.

**Why it matters:** deduplication is not a safe automatic operation here. Dropping all 24 risks deleting real observations and keeping them risks double counting. It also means no incremental refresh is possible, and these records cannot be joined to anything reported back later.

## Summary: the three problems

| Category | Problem | Evidence |
|---|---|---|
| Gaps | Missing values written as `?`, invisible to `isna()` | 4,262 cells across 2,399 rows, 7.4% |
| Strange values | `capital.gain` uses 99999 as a top-code sentinel | 159 rows, next real value 41,310, all `>50K` |
| Duplicates | Exact duplicate rows with no primary key to judge them | 24 rows, no identifier column, `fnlwgt` collides on 17,231 rows |